# Music Annotation Project V8.5 — Notebook 44
## Read-only treatment-control probability delta audit

**Runtime: CPU. Development-only diagnostic. No retraining, recalibration, threshold movement, or source mutation.**

Notebook 43 answered the registered operating-point question negatively: the corrected paired experiment compared **33×33 + 97×97** against the identical representation plus **193×193** outer context, but neither arm selected a group. Notebook 44 asks a narrower diagnostic question:

> **What did the 193×193 branch change in the model's group-level safety and value judgements, even though it did not create an operational selection?**

The audit therefore joins the two Notebook 43 OOF arms by the exact candidate identity and calculates paired deltas:

- `delta_p_value = treatment p_value - control p_value`
- `delta_p_safety = treatment p_safety - control p_safety`
- analogous raw-score and predicted-value deltas

It then separates candidates into interpretable quadrants, ranks the largest paired movements, produces candidate-centred contact sheets from the **observable annotated page only**, and reports post-hoc truth/value strata separately.

### Hard boundary

This notebook is **diagnostic only**. It must not:

- train or refit a model;
- fit/re-fit an isotonic calibrator;
- alter Notebook 43 probabilities;
- select a new probability threshold;
- create a challenger or promotion decision;
- alter any V7.8/V8.2/Notebook 43 source artefact;
- use clean truth in a visual panel presented as inference evidence.

A favourable subgroup discovered here is a hypothesis for a later pre-registered experiment, not permission to change the existing rule.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q pandas numpy pyarrow matplotlib opencv-python-headless

## 1. Configuration, source resolution and read-only assertions

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json, hashlib, math, shutil, sys, platform, zipfile
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt

ROOT=Path('/content/drive/MyDrive/MusicAnnotationProject')
N43_ROOT=ROOT/'observed_print_classifier_v8_5_notebook43_rebuilt'
OUTPUT_ROOT=ROOT/'observed_print_classifier_v8_5_notebook44_delta_audit'
SOURCE_N43_RUN_OVERRIDE=None
SEED=44001
TOP_N=60
CONTACT_N_PER_QUEUE=24
CROP_RADIUS=150
DELTA_EPS=1e-12
CONTROL_ARM='control_33_97'
TREATMENT_ARM='treatment_33_97_193'
EXPECTED_ARMS={CONTROL_ARM,TREATMENT_ARM}

REQUIRED=[
 '01_pre_registration.json','candidate_lock.json','SELF_VALIDATION_PASSED.json',
 'experiment_manifest.json','notebook43_verdict.json','executive_summary.txt',
 'tables/02_candidate_group_manifest.parquet','tables/05_counterfactual_value_labels.parquet',
 'tables/04_oof_safety_and_value_predictions.parquet','tables/arm_comparison_summary.csv',
 'tables/results_by_page.csv','tables/results_by_source_group.csv',
 'audit/model_inventory.csv','audit/calibrator_inventory.csv','audit/page_output_integrity.csv'
]

def sha256_file(p):
    h=hashlib.sha256()
    with open(p,'rb') as f:
        for b in iter(lambda:f.read(1048576),b''):h.update(b)
    return h.hexdigest()

def complete_runs(root):
    hits=[]
    if root.exists():
        for p in root.iterdir():
            if p.is_dir() and all((p/r).exists() for r in REQUIRED): hits.append(p)
    return sorted(hits,key=lambda p:p.stat().st_mtime)

if SOURCE_N43_RUN_OVERRIDE is not None:
    SOURCE=Path(SOURCE_N43_RUN_OVERRIDE)
    missing=[r for r in REQUIRED if not (SOURCE/r).exists()]
    if missing: raise RuntimeError(f'Notebook 43 source override incomplete: {missing}')
else:
    hits=complete_runs(N43_ROOT)
    if not hits: raise RuntimeError(f'No complete rebuilt Notebook 43 run found below {N43_ROOT}')
    SOURCE=hits[-1]

pre=json.loads((SOURCE/'01_pre_registration.json').read_text())
selfv=json.loads((SOURCE/'SELF_VALIDATION_PASSED.json').read_text())
verdict=json.loads((SOURCE/'notebook43_verdict.json').read_text())

# Fail if the attached run is not the corrected paired design.
pre_text=json.dumps(pre,sort_keys=True)
if '33+97' not in pre_text or '33+97+193' not in pre_text:
    raise RuntimeError('Notebook 43 pre-registration is not the corrected 33+97 versus 33+97+193 design.')
if not all(bool(v) for v in selfv.values() if isinstance(v,bool)):
    raise RuntimeError(f'Notebook 43 self-validation contains a false check: {selfv}')

RUN=datetime.now().strftime('%Y%m%d_%H%M%S')+'_v8_5_notebook44_delta_audit'
OUT=OUTPUT_ROOT/RUN; TABLES=OUT/'tables'; VISUAL=OUT/'visuals'; REVIEW=OUT/'review'; AUDIT=OUT/'audit'; UPLOAD=OUT/'upload'
for p in [OUT,TABLES,VISUAL,REVIEW,AUDIT,UPLOAD]:p.mkdir(parents=True,exist_ok=False)

print('Notebook 43 source:',SOURCE)
print('Notebook 43 verdict:',verdict.get('verdict'),'first failure:',verdict.get('first_failed_gate'))
print('Notebook 44 output:',OUT)

## 2. Load Notebook 43 OOF outputs and prove exact paired identity

In [ ]:
oof=pd.read_parquet(SOURCE/'tables'/'04_oof_safety_and_value_predictions.parquet')
cand=pd.read_parquet(SOURCE/'tables'/'02_candidate_group_manifest.parquet')
labels=pd.read_parquet(SOURCE/'tables'/'05_counterfactual_value_labels.parquet')
arm_summary=pd.read_csv(SOURCE/'tables'/'arm_comparison_summary.csv')

required_oof={'arm','row_id','page','source_group','candidate_group_id','mask_component_id','raw_safety','raw_value','p_safety','p_value','pred_value','safety_label','safety_label_eligible','value_positive','true_print','false_annotation','truth_uncertain','value_score'}
missing=required_oof-set(oof.columns)
if missing:raise RuntimeError(f'Notebook 43 OOF table missing required columns: {sorted(missing)}')
if set(oof.arm.astype(str).unique())!=EXPECTED_ARMS:raise RuntimeError(f'Unexpected arms: {sorted(oof.arm.unique())}')
if oof.duplicated(['arm','row_id']).any():raise RuntimeError('Duplicate arm/row_id pairs in Notebook 43 OOF output.')

arms={a:oof[oof.arm==a].copy().sort_values('row_id') for a in EXPECTED_ARMS}
for a,d in arms.items():
    if d.row_id.duplicated().any():raise RuntimeError(f'{a}: row_id not unique')
if list(arms[CONTROL_ARM].row_id)!=list(arms[TREATMENT_ARM].row_id):raise RuntimeError('Control/treatment candidate row IDs do not match exactly.')

identity_cols=['row_id','page','source_group','candidate_group_id','mask_component_id','safety_label','safety_label_eligible','value_positive','true_print','false_annotation','truth_uncertain','value_score']
left=arms[CONTROL_ARM][identity_cols].reset_index(drop=True)
right=arms[TREATMENT_ARM][identity_cols].reset_index(drop=True)
for c in identity_cols:
    if c=='row_id':continue
    if not left[c].equals(right[c]):raise RuntimeError(f'Paired identity mismatch in {c}')

pair=left.copy()
for prefix,arm in [('control',CONTROL_ARM),('treatment',TREATMENT_ARM)]:
    src=arms[arm].set_index('row_id')
    for c in ['raw_safety','raw_value','p_safety','p_value','pred_value','eligible_select','selected','reason']:
        if c in src.columns:pair[prefix+'_'+c]=pair.row_id.map(src[c])

for c in ['p_safety','p_value','raw_safety','raw_value','pred_value']:
    pair['delta_'+c]=pair['treatment_'+c]-pair['control_'+c]

pair['abs_delta_p_value']=pair.delta_p_value.abs()
pair['abs_delta_p_safety']=pair.delta_p_safety.abs()
pair['move_l2']=np.sqrt(pair.delta_p_value**2+pair.delta_p_safety**2)
pair['quadrant']=np.select([
    (pair.delta_p_value>DELTA_EPS)&(pair.delta_p_safety>DELTA_EPS),
    (pair.delta_p_value>DELTA_EPS)&(pair.delta_p_safety<-DELTA_EPS),
    (pair.delta_p_value<-DELTA_EPS)&(pair.delta_p_safety>DELTA_EPS),
    (pair.delta_p_value<-DELTA_EPS)&(pair.delta_p_safety<-DELTA_EPS),
],[
    'VALUE_UP__SAFETY_UP','VALUE_UP__SAFETY_DOWN','VALUE_DOWN__SAFETY_UP','VALUE_DOWN__SAFETY_DOWN'
],default='NO_MATERIAL_DIRECTION')
pair['safety_truth']=np.select([
    ~pair.safety_label_eligible.astype(bool),
    pair.safety_label.astype(bool)
],['TRUTH_UNCERTAIN','SAFE'],default='UNSAFE')
pair['value_truth']=np.where(pair.value_positive.astype(bool),'VALUE_POSITIVE','VALUE_NONPOSITIVE')
pair['truth_combo']=pair.safety_truth+'__'+pair.value_truth
pair.to_parquet(TABLES/'paired_candidate_deltas.parquet',index=False)
pair.to_csv(TABLES/'paired_candidate_deltas.csv',index=False)

identity={'source_run':str(SOURCE),'source_oof_sha256':sha256_file(SOURCE/'tables'/'04_oof_safety_and_value_predictions.parquet'),'pairs':len(pair),'arms':sorted(EXPECTED_ARMS),'row_ids_exact':True,'identity_columns_exact':True,'modified_source_files':False}
(OUT/'source_identity.json').write_text(json.dumps(identity,indent=2))
print(json.dumps(identity,indent=2))

## 3. Delta distribution, quadrants and truth/value stratification

In [ ]:
def qstats(x):
    x=pd.Series(x).dropna().astype(float)
    if x.empty:return {'n':0}
    qs=x.quantile([0,.01,.05,.10,.25,.5,.75,.9,.95,.99,1])
    return {'n':len(x),'mean':x.mean(),'sd':x.std(ddof=1),'min':qs.loc[0.0],'p01':qs.loc[.01],'p05':qs.loc[.05],'p10':qs.loc[.10],'p25':qs.loc[.25],'median':qs.loc[.5],'p75':qs.loc[.75],'p90':qs.loc[.9],'p95':qs.loc[.95],'p99':qs.loc[.99],'max':qs.loc[1.0]}

summ=[]
for metric in ['delta_p_value','delta_p_safety','delta_pred_value','delta_raw_value','delta_raw_safety']:
    summ.append({'population':'ALL','metric':metric,**qstats(pair[metric])})
    for key,g in pair.groupby('truth_combo'):
        summ.append({'population':key,'metric':metric,**qstats(g[metric])})
pd.DataFrame(summ).to_csv(TABLES/'delta_distribution_summary.csv',index=False)

quadrant=(pair.groupby(['quadrant','safety_truth','value_truth'],as_index=False).agg(
    groups=('row_id','size'),mean_delta_value=('delta_p_value','mean'),mean_delta_safety=('delta_p_safety','mean'),
    median_delta_value=('delta_p_value','median'),median_delta_safety=('delta_p_safety','median'),
    mean_value_score=('value_score','mean'),sum_true_print=('true_print','sum'),sum_false_annotation=('false_annotation','sum')
))
quadrant.to_csv(TABLES/'quadrant_truth_summary.csv',index=False)

page=(pair.groupby(['page','source_group'],as_index=False).agg(
    groups=('row_id','size'),mean_delta_value=('delta_p_value','mean'),mean_delta_safety=('delta_p_safety','mean'),
    median_delta_value=('delta_p_value','median'),median_delta_safety=('delta_p_safety','median'),
    value_up_safety_down=('quadrant',lambda s:int((s=='VALUE_UP__SAFETY_DOWN').sum())),
    both_up=('quadrant',lambda s:int((s=='VALUE_UP__SAFETY_UP').sum())),move_l2_sum=('move_l2','sum')
))
page.to_csv(TABLES/'delta_summary_by_page.csv',index=False)

group=(pair.groupby('source_group',as_index=False).agg(
    groups=('row_id','size'),mean_delta_value=('delta_p_value','mean'),mean_delta_safety=('delta_p_safety','mean'),
    value_up_safety_down=('quadrant',lambda s:int((s=='VALUE_UP__SAFETY_DOWN').sum())),
    both_up=('quadrant',lambda s:int((s=='VALUE_UP__SAFETY_UP').sum())),move_l2_sum=('move_l2','sum')
))
group.to_csv(TABLES/'delta_summary_by_source_group.csv',index=False)

display(pd.DataFrame(summ).query("population=='ALL'"))
display(quadrant.sort_values('groups',ascending=False))

## 4. Ranking diagnostics: does 193 context move valuable or safe groups in the right direction?

In [ ]:
# No threshold is selected here. These are descriptive paired-ranking diagnostics only.
def ranking_row(label,mask):
    q=pair[mask].copy()
    return {
        'stratum':label,'groups':len(q),
        'mean_delta_p_value':q.delta_p_value.mean() if len(q) else np.nan,
        'median_delta_p_value':q.delta_p_value.median() if len(q) else np.nan,
        'fraction_value_up':float((q.delta_p_value>0).mean()) if len(q) else np.nan,
        'mean_delta_p_safety':q.delta_p_safety.mean() if len(q) else np.nan,
        'median_delta_p_safety':q.delta_p_safety.median() if len(q) else np.nan,
        'fraction_safety_up':float((q.delta_p_safety>0).mean()) if len(q) else np.nan,
        'fraction_value_up_safety_down':float((q.quadrant=='VALUE_UP__SAFETY_DOWN').mean()) if len(q) else np.nan,
    }
rows=[
 ranking_row('ALL',np.ones(len(pair),bool)),
 ranking_row('VALUE_POSITIVE',pair.value_positive.astype(bool)),
 ranking_row('VALUE_NONPOSITIVE',~pair.value_positive.astype(bool)),
 ranking_row('SAFE',pair.safety_truth=='SAFE'),
 ranking_row('UNSAFE',pair.safety_truth=='UNSAFE'),
 ranking_row('TRUTH_UNCERTAIN',pair.safety_truth=='TRUTH_UNCERTAIN'),
 ranking_row('SAFE_VALUE_POSITIVE',(pair.safety_truth=='SAFE')&pair.value_positive.astype(bool)),
 ranking_row('UNSAFE_VALUE_POSITIVE',(pair.safety_truth=='UNSAFE')&pair.value_positive.astype(bool)),
]
rankdiag=pd.DataFrame(rows)
rankdiag.to_csv(TABLES/'paired_ranking_diagnostics.csv',index=False)
display(rankdiag)

## 5. Predeclared review queues

In [ ]:
queues={
 '01_value_up_safety_down':pair[pair.quadrant=='VALUE_UP__SAFETY_DOWN'].sort_values(['delta_p_value','abs_delta_p_safety'],ascending=[False,False]),
 '02_both_up':pair[pair.quadrant=='VALUE_UP__SAFETY_UP'].sort_values('move_l2',ascending=False),
 '03_safety_down_largest':pair.sort_values('delta_p_safety').head(TOP_N),
 '04_safety_up_largest':pair.sort_values('delta_p_safety',ascending=False).head(TOP_N),
 '05_value_up_largest':pair.sort_values('delta_p_value',ascending=False).head(TOP_N),
 '06_value_down_largest':pair.sort_values('delta_p_value').head(TOP_N),
 '07_largest_joint_movement':pair.sort_values('move_l2',ascending=False).head(TOP_N),
 '08_safe_value_positive_movements':pair[(pair.safety_truth=='SAFE')&pair.value_positive.astype(bool)].sort_values('move_l2',ascending=False).head(TOP_N),
 '09_unsafe_value_positive_movements':pair[(pair.safety_truth=='UNSAFE')&pair.value_positive.astype(bool)].sort_values('move_l2',ascending=False).head(TOP_N),
}
for name,q in queues.items():q.head(TOP_N).to_csv(TABLES/f'{name}.csv',index=False)
queue_summary=pd.DataFrame([{'queue':k,'available_groups':len(v),'exported_groups':min(len(v),TOP_N)} for k,v in queues.items()])
queue_summary.to_csv(TABLES/'review_queue_summary.csv',index=False)
display(queue_summary)

## 6. Observable candidate-centred contact sheets

These contact sheets deliberately use the aligned/annotated **observable page**, not the clean page. Clean truth remains in the tables as post-hoc labels only.

The red rectangle marks the Notebook 43 candidate-group bounding box. Captions display candidate identity plus treatment-minus-control probability deltas. They do **not** display an inferred semantic failure-mode label.

In [ ]:
obs_dir=ROOT/'aligned'
obs_map={p.stem:p for p in obs_dir.iterdir() if p.is_file() and p.suffix.lower() in {'.png','.jpg','.jpeg','.tif','.tiff','.bmp'}}
manifest=cand.copy()
if 'row_id' not in manifest.columns:
    manifest=manifest.reset_index().rename(columns={'index':'row_id'})
manifest_lookup=manifest.set_index('row_id')

def bbox_for(row):
    r=manifest_lookup.loc[int(row.row_id)]
    # Prefer explicit candidate bounding-box fields if present.
    candidates=[('x','y','w','h'),('bbox_x','bbox_y','bbox_w','bbox_h')]
    for names in candidates:
        if all(n in r.index and pd.notna(r[n]) for n in names):
            x,y,w,h=[int(r[n]) for n in names];return x,y,w,h
    # Fallback to centre and a visible box if the manifest only stores centres.
    cx=int(r['center_x']) if 'center_x' in r.index else int(r['x'])
    cy=int(r['center_y']) if 'center_y' in r.index else int(r['y'])
    return cx-4,cy-4,9,9

def crop_panel(row):
    page=str(row.page)
    if page not in obs_map:return None
    im=cv2.imread(str(obs_map[page]),0)
    if im is None:return None
    x,y,w,h=bbox_for(row); H,W=im.shape;cx=x+w//2;cy=y+h//2
    x0=max(0,cx-CROP_RADIUS);x1=min(W,cx+CROP_RADIUS+1);y0=max(0,cy-CROP_RADIUS);y1=min(H,cy+CROP_RADIUS+1)
    crop=cv2.cvtColor(im[y0:y1,x0:x1],cv2.COLOR_GRAY2RGB)
    bx0=max(0,x-x0);by0=max(0,y-y0);bx1=min(crop.shape[1]-1,x+w-x0);by1=min(crop.shape[0]-1,y+h-y0)
    cv2.rectangle(crop,(bx0,by0),(bx1,by1),(255,0,0),2)
    return crop

def contact_sheet(q,name,n=CONTACT_N_PER_QUEUE,cols=4):
    q=q.head(n).reset_index(drop=True)
    if q.empty:return
    rows=math.ceil(len(q)/cols);fig,axes=plt.subplots(rows,cols,figsize=(5*cols,5*rows),squeeze=False)
    for ax in axes.flat:ax.axis('off')
    for i,r in q.iterrows():
        ax=axes.flat[i];im=crop_panel(r)
        if im is not None:ax.imshow(im)
        ax.set_title(f"{r.page} g{int(r.candidate_group_id)} | dV={r.delta_p_value:+.3f} dS={r.delta_p_safety:+.3f}\n{r.safety_truth} | {r.value_truth}",fontsize=9)
        ax.axis('off')
    plt.tight_layout();plt.savefig(REVIEW/f'{name}.png',dpi=150,bbox_inches='tight');plt.close(fig)

for name,q in queues.items():contact_sheet(q,name)
print('Observable contact sheets:',REVIEW)

## 7. Plots: paired probability movement

In [ ]:
plt.figure(figsize=(8,7))
colors={'SAFE':'#2ca02c','UNSAFE':'#d62728','TRUTH_UNCERTAIN':'#ffbf00'}
for k,g in pair.groupby('safety_truth'):
    plt.scatter(g.delta_p_value,g.delta_p_safety,s=22,alpha=.65,label=k,color=colors.get(k))
plt.axhline(0,color='black',lw=1);plt.axvline(0,color='black',lw=1)
plt.xlabel('Treatment - control calibrated value probability')
plt.ylabel('Treatment - control calibrated safety probability')
plt.title('Notebook 44: effect of adding 193x193 outer context')
plt.legend();plt.tight_layout();plt.savefig(VISUAL/'delta_scatter_by_safety_truth.png',dpi=180);plt.close()

fig,axes=plt.subplots(1,2,figsize=(14,5))
axes[0].hist(pair.delta_p_value,bins=60);axes[0].axvline(0,color='black');axes[0].set_title('Delta calibrated value probability');axes[0].set_xlabel('Treatment - control')
axes[1].hist(pair.delta_p_safety,bins=60);axes[1].axvline(0,color='black');axes[1].set_title('Delta calibrated safety probability');axes[1].set_xlabel('Treatment - control')
plt.tight_layout();plt.savefig(VISUAL/'delta_histograms.png',dpi=180);plt.close()

q=pair.assign(value_positive_name=np.where(pair.value_positive,'value positive','value non-positive'))
fig,axes=plt.subplots(1,2,figsize=(14,5))
q.boxplot(column='delta_p_value',by='value_positive_name',ax=axes[0]);axes[0].set_title('Value delta by value truth');axes[0].set_xlabel('');axes[0].set_ylabel('Treatment - control p_value')
q.boxplot(column='delta_p_safety',by='safety_truth',ax=axes[1]);axes[1].set_title('Safety delta by safety truth');axes[1].set_xlabel('');axes[1].set_ylabel('Treatment - control p_safety')
plt.suptitle('');plt.tight_layout();plt.savefig(VISUAL/'delta_truth_boxplots.png',dpi=180);plt.close()
print('Plots:',VISUAL)

## 8. Read-only conclusions, integrity record and diagnostic verdict

In [ ]:
# Notebook 44 does not invent a pass/fail promotion gate. It emits a diagnostic classification.
a=arm_summary.set_index('arm')
control=a.loc[CONTROL_ARM]; treatment=a.loc[TREATMENT_ARM]
value_ap_delta=float(treatment.value_average_precision-control.value_average_precision)
safety_ap_delta=float(treatment.safety_average_precision-control.safety_average_precision)

# Descriptive directional evidence only.
vp=pair[pair.value_positive.astype(bool)]
safe=pair[pair.safety_truth=='SAFE']
unsafe=pair[pair.safety_truth=='UNSAFE']
quad_counts=pair.quadrant.value_counts().to_dict()

if int(treatment.selected_groups)==0 and int(control.selected_groups)==0:
    operating='NO_OPERATIONAL_SELECTION_IN_EITHER_ARM'
else:
    operating='SELECTION_OCCURRED_REVIEW_NOTEBOOK43_RESULTS'

if value_ap_delta>0 and safety_ap_delta<0:
    pattern='VALUE_RANKING_UP__SAFETY_RANKING_DOWN'
elif value_ap_delta>0 and safety_ap_delta>=0:
    pattern='BOTH_RANKING_METRICS_UP_OR_FLAT'
elif value_ap_delta<=0 and safety_ap_delta>0:
    pattern='SAFETY_RANKING_UP__VALUE_RANKING_DOWN_OR_FLAT'
else:
    pattern='NO_RANKING_IMPROVEMENT'

diagnostic={
 'notebook':'44','read_only':True,'source_notebook43_run':str(SOURCE),
 'paired_groups':int(len(pair)),'operating_point_status':operating,
 'control_value_average_precision':float(control.value_average_precision),'treatment_value_average_precision':float(treatment.value_average_precision),'delta_value_average_precision':value_ap_delta,
 'control_safety_average_precision':float(control.safety_average_precision),'treatment_safety_average_precision':float(treatment.safety_average_precision),'delta_safety_average_precision':safety_ap_delta,
 'observed_ranking_pattern':pattern,
 'mean_delta_p_value_all':float(pair.delta_p_value.mean()),'mean_delta_p_safety_all':float(pair.delta_p_safety.mean()),
 'mean_delta_p_value_value_positive':float(vp.delta_p_value.mean()) if len(vp) else None,
 'mean_delta_p_safety_safe':float(safe.delta_p_safety.mean()) if len(safe) else None,
 'mean_delta_p_safety_unsafe':float(unsafe.delta_p_safety.mean()) if len(unsafe) else None,
 'quadrant_counts':quad_counts,
 'new_threshold_selected':False,'model_fit':False,'calibrator_fit':False,'source_files_modified':False,
 'interpretation_limit':'Association of the added 193x193 branch with paired prediction deltas; no semantic ownership mechanism is inferred from probabilities alone.'
}
(OUT/'diagnostic_summary.json').write_text(json.dumps(diagnostic,indent=2))

integrity={
 'paired_row_ids_exact':True,'paired_identity_columns_exact':True,'source_oof_sha256':sha256_file(SOURCE/'tables'/'04_oof_safety_and_value_predictions.parquet'),
 'source_arm_summary_sha256':sha256_file(SOURCE/'tables'/'arm_comparison_summary.csv'),'no_retraining':True,'no_recalibration':True,'no_threshold_search':True,
 'visuals_use_observable_aligned_pages':True,'clean_truth_not_rendered_in_contact_sheets':True,'audit_is_development_only':True
}
(OUT/'READ_ONLY_INTEGRITY.json').write_text(json.dumps(integrity,indent=2))

lines=[
 'NOTEBOOK 44 READ-ONLY TREATMENT-CONTROL DELTA AUDIT','',
 f'Source Notebook 43: {SOURCE}',f'Paired candidate groups: {len(pair)}',
 f'Control value AP: {control.value_average_precision:.6f}',f'Treatment value AP: {treatment.value_average_precision:.6f}',f'Delta value AP: {value_ap_delta:+.6f}',
 f'Control safety AP: {control.safety_average_precision:.6f}',f'Treatment safety AP: {treatment.safety_average_precision:.6f}',f'Delta safety AP: {safety_ap_delta:+.6f}',
 f'Notebook 43 operational status: {operating}',f'Observed ranking pattern: {pattern}','',
 'Paired probability delta quadrants:'
]
for k,v in sorted(quad_counts.items()):lines.append(f'- {k}: {v}')
lines += ['',
 'Interpretation boundary:',
 '- This is a read-only diagnostic of existing Notebook 43 OOF predictions.',
 '- No threshold was tuned and no new selection policy was created.',
 '- The audit can show where 193x193 context changed value/safety confidence; it cannot by itself prove why the network changed its judgement.',
 '- Review the value-up/safety-down queue first, then both-up and the largest safety regressions.',
]
(OUT/'executive_summary.txt').write_text(chr(10).join(lines))
print(chr(10).join(lines))

## 9. Final upload package

In [ ]:
# Curated review package. Heavy source artefacts remain in Notebook 43 and are referenced by identity/hash.
required=[
 OUT/'executive_summary.txt',OUT/'diagnostic_summary.json',OUT/'READ_ONLY_INTEGRITY.json',OUT/'source_identity.json',
 TABLES/'paired_candidate_deltas.csv',TABLES/'delta_distribution_summary.csv',TABLES/'quadrant_truth_summary.csv',TABLES/'paired_ranking_diagnostics.csv',
 TABLES/'delta_summary_by_page.csv',TABLES/'delta_summary_by_source_group.csv',TABLES/'review_queue_summary.csv',
 VISUAL/'delta_scatter_by_safety_truth.png',VISUAL/'delta_histograms.png',VISUAL/'delta_truth_boxplots.png'
]
required += sorted(REVIEW.glob('*.png'))
missing=[str(p) for p in required if not p.exists()]
if missing:raise RuntimeError(f'Notebook 44 upload package missing: {missing}')

inventory=[]
for p in required:
 rel=p.relative_to(OUT);dst=UPLOAD/rel;dst.parent.mkdir(parents=True,exist_ok=True);shutil.copy2(p,dst)
 inventory.append({'relative_path':str(rel),'bytes':p.stat().st_size,'sha256':sha256_file(p)})
pd.DataFrame(inventory).to_csv(UPLOAD/'upload_manifest.csv',index=False)
(UPLOAD/'upload_file_list.txt').write_text(chr(10).join(x['relative_path'] for x in inventory))
(UPLOAD/'README.txt').write_text('Notebook 44 is a read-only delta audit of the corrected Notebook 43 paired 33+97 versus 33+97+193 experiment. Start with executive_summary.txt, diagnostic_summary.json, paired_ranking_diagnostics.csv, quadrant_truth_summary.csv and review/01_value_up_safety_down.png. No model/calibrator/threshold was changed.')
zip_path=UPLOAD/'notebook44_read_only_delta_audit_upload_bundle.zip'
with zipfile.ZipFile(zip_path,'w',zipfile.ZIP_DEFLATED) as z:
    for p in UPLOAD.rglob('*'):
        if p.is_file() and p!=zip_path:z.write(p,p.relative_to(UPLOAD))
print('Notebook 44 complete:',OUT)
print('Upload bundle:',zip_path)